---
## Section 1: Connections

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Gemini key loaded:   ", "✅" if GEMINI_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Gemini key loaded:    ✅


In [2]:
from pageindex import PageIndexClient
import google.generativeai as genai


pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-3.1-flash-lite-preview")

print("✅ PageIndex client ready")
print("✅ Gemini client ready")

✅ PageIndex client ready
✅ Gemini client ready


/home/spy/miniforge3/envs/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Section 2: Upload & Index a PDF

In [3]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./attention_all_you_need.pdf"   # ← change this

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

Uploading: ./attention_all_you_need.pdf
✅ Uploaded!
📋 Document ID: pi-cmo5wys6w12b101r4j9ofr7rf
   (Save this ID — you'll use it throughout the notebook)


In [4]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("Building tree index...")
print("(This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

Building tree index...
(This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [5]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*\u2020 University of Toronto aidan@cs.toronto.edu | \u0141ukasz Kaiser* Google Brain lukaszkaiser@google.com  |   |\n\nIllia Polosukhin*\u2020\nillia.polosukhin@gmail.com\n",
  "text": "# Attention Is All You Need\n\n|  Ashish Vaswani* Google Brain avaswani@google.com | Noam Shazeer* Google Brain noam@google.com | Niki Parmar* Google Research nikip@google.com | Jakob Uszkoreit* Google Research usz@google.com  |\n| --- | --- | --- | --- |\n|  Llion Jones* Google Research llion@google.com | Aidan N. Gomez*\u2020 Universi

In [6]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Attention Is All You Need  (p.1)
  └─ [0001] Abstract  (p.1)
  └─ [0002] 2 Background  (p.2)
  └─ [0003] 3 Model Architecture  (p.2)
    └─ [0004] 3.1 Encoder and Decoder Stacks  (p.3)
    └─ [0005] 3.2 Attention  (p.3)
      └─ [0006] 3.2.1 Scaled Dot-Product Attention  (p.4)
      └─ [0007] 3.2.2 Multi-Head Attention  (p.4)
      └─ [0008] 3.2.3 Applications of Attention in our Model  (p.5)
    └─ [0009] 3.3 Position-wise Feed-Forward Networks  (p.5)
    └─ [0010] 3.4 Embeddings and Softmax  (p.5)
    └─ [0011] 3.5 Positional Encoding  (p.6)
  └─ [0012] 4 Why Self-Attention  (p.6)
  └─ [0013] 5 Training  (p.7)
  └─ [0014] 6 Results  (p.8)
    └─ [0015] 6.1 Machine Translation  (p.8)
    └─ [0016] 6.2 Model Variations  (p.8)
    └─ [0017] 6.3 English Constituency Parsing  (p.9)
  └─ [0018] 7 Conclusion  (p.10)
  └─ [0019] References  (p.10)
  └─ [0020] Attention Visualizations  (p.13)


In [7]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 21
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [8]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    global model
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    response = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json"
        }
    )
    
    # Gemini returns text directly
    return json.loads(response.text)

In [9]:
# ── Test with a sample query (NOT related to attention all you need)─────────────────────────────────────────────────
query = "What is the syllabus covered in Modern LLM finetuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in Modern LLM finetuning?

🧠 LLM Reasoning:
The user is asking for the syllabus covered in 'Modern LLM finetuning'. I have reviewed the document tree, which represents the paper 'Attention Is All You Need'. This paper describes the Transformer architecture and does not contain information regarding a syllabus for a course or tutorial on 'Modern LLM finetuning'. Therefore, no nodes in the provided document structure are relevant to the query.

🎯 Selected Node IDs: []


In [10]:
# ── Test with a sample query (NOT related to attention all you need)─────────────────────────────────────────────────
query = "What is the main architectural innovation of the Transformer model, and how does it differ from previous sequence transduction models?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the main architectural innovation of the Transformer model, and how does it differ from previous sequence transduction models?

🧠 LLM Reasoning:
The user is asking about the main architectural innovation of the Transformer (attention) and how it differs from previous recurrent or convolutional models. Node '0001' (Abstract) provides an overview of the difference from traditional models. Node '0003' (Model Architecture) describes the overall structure. Node '0005' (Attention) details the core innovation, with its sub-nodes '0006', '0007', and '0008' explaining the specific implementation. Node '0012' (Why Self-Attention) directly contrasts the Transformer's architecture with the recurrent and convolutional layers used in previous models, making it highly relevant.

🎯 Selected Node IDs: ['0001', '0003', '0005', '0012']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [11]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [12]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    global model
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json"
        }
    )
    
    # Gemini returns text directly
    return response.text

In [13]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [14]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the syllabus covered in modern llm finetuning?",
    tree=pageindex_tree
)

🔍 Query: What are the syllabus covered in modern llm finetuning?

🧠 Reasoning: The user is asking about the syllabus for modern LLM finetuning. I have examined the provided document structure, which is the 'Attention Is All You Need' paper (the original Transformer paper). This ...
🎯 Retrieved node IDs: []
📄 Sections found: []

📝 Answer:
⚠️ No relevant sections found in the document.


In [15]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What is the main architectural innovation of the Transformer model, and how does it differ from previous sequence transduction models?",
    "What is the formula for Scaled Dot-Product Attention in the Transformer?",
    "How are positional encodings defined in the Transformer model?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans}...")
    print("-" * 55)


Q: What is the main architectural innovation of the Transformer model, and how does it differ from previous sequence transduction models?
A: {
  "main_architectural_innovation": "The Transformer is a sequence transduction model based entirely on attention mechanisms, dispensing with recurrence and convolutions entirely (Abstract, Page 1).",
  "difference_from_previous_models": "Previous sequence transduction models typically rely on complex recurrent or convolutional neural networks to factor computation along symbol positions, which makes them inherently sequential and precludes parallelization (Abstract, Page 1; 1 Introduction, Page 1). In contrast, the Transformer uses multi-headed self-attention to connect all positions with a constant number of sequentially executed operations, allowing for significantly more parallelization and faster training (4 Why Self-Attention, Page 6; 7 Conclusion, Page 10)."
}...
-------------------------------------------------------

Q: What is the form

---
## 💬 Section 6: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [16]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the key findings from **"Attention Is All You Need"** (Vaswani et al., 2017):

---

### 🔑 Core Contribution
The paper introduces the **Transformer** — the first sequence transduction model built entirely on **self-attention**, completely discarding recurrence (RNNs/LSTMs) and convolutions.

---

### 📊 Machine Translation Results
- **English-to-German (WMT 2014):** Achieved **28.4 BLEU**, surpassing all prior models (including ensembles) by **>2 BLEU points**.
- **English-to-French (WMT 2014):** Achieved **41.8 BLEU** — a new single-model state-of-the-art — trained in just **3.5 days on 8 GPUs**, at less than ¼ the cost of prior best models.

---

### ⚡ Training Efficiency
The Transformer trains **significantly faster** than RNN/CNN-based architectures due to full parallelizability. Even the base model outperforms all prior published models at a fraction of the training cost.

---

### 🏗️ Architecture Insights (Model Variations)
- **8 attention heads** is the

In [17]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What is the Transformer architecture and what components does it consist of?",
    "How does the attention mechanism used in it work mathematically?",
    "Why is this approach more efficient than recurrent or convolutional models?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What is the Transformer architecture and what components does it consist of?
🤖 Assistant: ## The Transformer Architecture

The Transformer is a sequence transduction model that **relies entirely on attention mechanisms**, completely replacing recurrence (RNNs/LSTMs) and convolutions. It follows an **encoder-decoder** structure.

---

### High-Level Structure

- **Encoder**: Maps an input sequence $(x_1, \ldots, x_n)$ to continuous representations $\mathbf{z}$.
- **Decoder**: Auto-regre...
-------------------------------------------------------

👤 User: How does the attention mechanism used in it work mathematically?
🤖 Assistant: Here is a detailed mathematical breakdown of the attention mechanism from the paper:

---

## Attention Mechanism — Mathematical Walkthrough

### Core Concept
An attention function maps a **query** and a set of **key-value pairs** to an output. The output is a **weighted sum of the values**, where each weight reflects how compatible the query is with t